[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://github.com/codespaces)


<!-- gen:values.yml#weeks.08.notebook-title|Debrief -->
# Week 8 Debrief: DataViz II: Storytelling, decluttering, and accessibility
<!-- /gen -->

*Worked answers for [Quest 08](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest): The model slide, a model [design rationale](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-design-rationale), the [Red Flags](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-red-flags) post-mortem, and this week's [AI failure gallery](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-failure-gallery).*

⚠️ **Complete the quest first.** Several designs can answer the CFO well, provided you defend each choice. Mark your work against the *reasoning* here.

Run the cell below first.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
# The quest's generator, with the same seed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(5813)  # module seed: Do not change

dates = pd.date_range("2024-07-01", "2026-06-30", freq="D")
n = len(dates)
day = pd.DataFrame({"date": dates})
day["is_weekend"] = day["date"].dt.dayofweek >= 5

sat = day["date"].where(day["date"].dt.dayofweek == 5,
                        day["date"] - pd.Timedelta(days=1))
day["weekend_id"] = sat.where(day["is_weekend"])
saturdays = day.loc[day["is_weekend"], "weekend_id"].unique()
promo_weekends = rng.permutation(saturdays)[: len(saturdays) // 2]
day["promo"] = day["weekend_id"].isin(promo_weekends)

t = np.arange(n)
season = 6.0 * np.sin(2 * np.pi * t / 365.25 + 1.2)
base = np.where(day["is_weekend"], 58.0, 41.0)
fade = np.linspace(1.0, 0.25, n)
uplift = np.where(day["promo"], 9.5 * fade, 0.0)
day["net_takings"] = (base + season + uplift + rng.normal(0, 4.0, n)).round(1)
day["footfall"] = (day["net_takings"] * 32 + rng.normal(0, 150, n)).round().astype(int)

departments = ["Kitchen", "Living", "Garden", "Gifts", "Stationery", "Café"]
dept_share = np.array([0.30, 0.17, 0.21, 0.14, 0.10, 0.08])
weeks = pd.date_range("2024-07-06", "2026-06-27", freq="W-SAT")
wt = np.arange(len(weeks))
weekly_total = 2450 + 180 * np.sin(2 * np.pi * wt / 52.18 + 1.2) + rng.normal(0, 60, len(weeks))
weekly = pd.DataFrame(
    {"week": np.repeat(weeks, len(departments)),
     "department": np.tile(departments, len(weeks)),
     "revenue": (np.outer(weekly_total, dept_share)
                 * (1 + rng.normal(0, 0.07, (len(weeks), len(departments))))).round(1).ravel()}
)

print(f"Recreated day ({len(day)} days) and weekly ({len(weekly)} rows), seed 5813, identical to the quest")

## 🔍 First look

Judge each chart against the CFO's question: *Should we keep weekend promotions?* The promotion premium is the extra takings a promotion weekend earns over an ordinary one. The [Introduction](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/08/Introduction.ipynb), section "Showing how sure you are", explains the 95% bands.

- **Chart A**, a pie of department shares, answers a part-to-whole question and shows nothing about promotions or time. Six slices are also more than a pie can show clearly. Its only use is context on the business mix
- **Chart B**, weekly lines for six departments, shows change over time. It splits by department, though. The CFO's comparison is promotion weekends against ordinary weekends. Weekly points add fifty-two noisy ups and downs a year. Its only use is seeing seasonality
- **Chart C**, the footfall-against-takings scatter, shows a relationship. It is the most convincing of the three wrong charts. It hides time, so a fading premium and a stable one look identical. Its red-and-green colours are also the classic colour-blindness mistake
- **Chart D** plots quarterly means for promotion and ordinary weekends with 95% bands. This is the chart to choose. It compares promotion weekends with ordinary weekends, over time, at the quarterly level the board thinks in, and it shows uncertainty

[Bronze](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bronze) asks for one sentence. A good one makes these points about **Chart D**:

- It is the only chart that compares promotion against ordinary weekends over time, which is the comparison the <!-- plain-language: everyday decision --> decision is about
- It shows the uncertainty
- Its faults are all in the styling, and you can fix them

## 🥉 Bronze: The decluttered Chart D

The cell below rebuilds Chart D's quarterly averages from [First look](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-first-look) and prints the promotion premium (promotion minus ordinary) quarter by quarter. Read the numbers before you look at the charts. Note what happens after the winter of 2024-25.

In [ ]:
# Recreate Chart D's aggregation (as in the quest's First look)
wk = day[day["is_weekend"]].copy()
wk["quarter"] = wk["date"].dt.to_period("Q").dt.to_timestamp()
q = (wk.groupby(["quarter", "promo"])["net_takings"]
       .agg(mean="mean", sem="sem")
       .reset_index())
promo_line = q[q["promo"]].reset_index(drop=True)
base_line = q[~q["promo"]].reset_index(drop=True)

# The premium the board is deciding about, quarter by quarter:
gap = (promo_line["mean"] - base_line["mean"]).round(1)
print("Promotion premium by quarter (£k/day):", list(gap))

The cell below applies the whole declutter sequence ([design guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md), section "3. R: Reduce") to Chart D. It adds no annotation or takeaway title yet, because you add those in [Silver](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-silver). Both 95% bands stay.

In [ ]:
# 🥉 Bronze model: Each numbered comment names its declutter step
ACCENT, CONTEXT, TEXT_GREY = "#0072B2", "#767676", "#767676"

fig, ax = plt.subplots(figsize=(9.5, 5))
for line, colour, lw in [(promo_line, ACCENT, 2.4), (base_line, CONTEXT, 1.6)]:
    ax.plot(line["quarter"], line["mean"], marker="o", markersize=4,
            color=colour, linewidth=lw)                                   # 6 emphasis
    ax.fill_between(line["quarter"],
                    line["mean"] - 1.96 * line["sem"],
                    line["mean"] + 1.96 * line["sem"],
                    color=colour, alpha=0.15, linewidth=0)                # bands kept

ax.annotate("Promotion weekends", (promo_line["quarter"].iloc[-1],
            promo_line["mean"].iloc[-1]), xytext=(10, 10),
            textcoords="offset points", va="center", fontsize=9,
            color=ACCENT, fontweight="bold")                              # 4 direct labels
ax.annotate("Ordinary weekends", (base_line["quarter"].iloc[-1],
            base_line["mean"].iloc[-1]), xytext=(10, -14),
            textcoords="offset points", va="center", fontsize=9, color=TEXT_GREY)

ax.spines[["top", "right"]].set_visible(False)                            # 1 spines
ax.grid(axis="y", color="#e0e0e0", linewidth=0.8)                         # 2 gridlines
ax.set_axisbelow(True)                                                    # 3 background (white by default)
ax.set_ylabel("Mean weekend net takings per store (£k)")                  # 5 units
ax.yaxis.set_major_locator(plt.MultipleLocator(5))                        # 5 rounded ticks
ax.margins(x=0.03)
ax.set_title("Mean weekend net takings by quarter, promotion vs ordinary weekends",
             loc="left")
plt.tight_layout()
plt.show()

The figure is less favourable to promotions than the Trading team claims. Look at the gap between the lines since early 2025, and at how wide the bands are compared with that gap.

## 🥈 Silver: The model slide and the model rationale

The premium printed above is roughly **£12–14k per store-day** through the winter of 2024-25. It then falls almost every quarter. By spring 2026 it is **under £2k, inside the overlap of the 95% bands**. So the takeaway title is that the premium is disappearing. The cell below builds the slide that shows the board exactly that, and prints its alt text.

In [ ]:
# 🥈 Silver model: The boardroom slide.
ACCENT, CONTEXT, TEXT_GREY = "#0072B2", "#767676", "#767676"

fig, ax = plt.subplots(figsize=(10, 5.6))
for line, colour, lw in [(promo_line, ACCENT, 2.4), (base_line, CONTEXT, 1.6)]:
    ax.plot(line["quarter"], line["mean"], marker="o", markersize=4,
            color=colour, linewidth=lw)
    ax.fill_between(line["quarter"],
                    line["mean"] - 1.96 * line["sem"],
                    line["mean"] + 1.96 * line["sem"],
                    color=colour, alpha=0.15, linewidth=0)

# Place the direct labels in clear space beside each line
ax.annotate("Promotion weekends", (promo_line["quarter"].iloc[4],
            promo_line["mean"].iloc[4]), xytext=(0, 40),
            textcoords="offset points", ha="center", fontsize=9.5,
            color=ACCENT, fontweight="bold")
ax.annotate("Ordinary weekends", (base_line["quarter"].iloc[1],
            base_line["mean"].iloc[1]), xytext=(0, -30),
            textcoords="offset points", ha="center", fontsize=9.5, color=TEXT_GREY)

# Mark the premium as the gap between the two lines, early and late
for i in (2, -1):
    x = promo_line["quarter"].iloc[i]
    ax.annotate("", xy=(x, promo_line["mean"].iloc[i]), xytext=(x, base_line["mean"].iloc[i]),
                arrowprops=dict(arrowstyle="<->", color="#444444", linewidth=1,
                                shrinkA=3, shrinkB=3))
early_mid = (promo_line["mean"].iloc[2] + base_line["mean"].iloc[2]) / 2
ax.annotate("premium ≈\n£12–14k/day\n(winter 2024-25)", (promo_line["quarter"].iloc[2], early_mid),
            xytext=(-8, 0), textcoords="offset points", ha="right", va="center",
            fontsize=9, color="#444444")
late_mid = (promo_line["mean"].iloc[-1] + base_line["mean"].iloc[-1]) / 2
ax.annotate("≈ £2k/day, inside the\nmargin of error", (promo_line["quarter"].iloc[-1], late_mid),
            xytext=(-30, 70), textcoords="offset points", ha="right", va="center",
            fontsize=9, color="#444444",
            arrowprops=dict(arrowstyle="->", color=TEXT_GREY, shrinkB=4))

ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e0e0e0", linewidth=0.8)
ax.set_axisbelow(True)
ax.set_ylabel("Mean weekend net takings per store (£k)")
ax.yaxis.set_major_locator(plt.MultipleLocator(5))
ax.margins(x=0.03)

ax.set_title("The weekend promotion premium has almost disappeared",
             loc="left", fontsize=13.5, fontweight="bold", pad=30)
ax.text(0, 1.04, "Mean net takings per store on weekend days (£k), quarterly with 95% bands · promotion vs ordinary weekends · Jul 2024–Jun 2026",
        transform=ax.transAxes, fontsize=9, color="#555555")
fig.text(0.07, -0.02,
         "Source: Marlow & Finch till data, Jul 2024–Jun 2026; takings net of promotional markdowns; all 12 stores.",
         fontsize=8, color=TEXT_GREY)
plt.tight_layout()
plt.show()

alt_text = (
    "Line chart of quarterly mean weekend net takings per store, promotion versus "
    "ordinary weekends with 95% confidence bands, July 2024 to June 2026, showing "
    "the promotion premium shrinking from £12–14k per store-day in winter 2024-25 "
    "to under £2k (within the margin of error) by spring 2026."
)
print("Alt text:", alt_text)

The slide's answer is a qualified one: *"The premium is vanishing; do not renew it automatically."* The figure supports a fair test, such as a promotion-free month in the autumn, before the company recommits the markdown budget.

### The model rationale (241 words)

> **Slide: The weekend promotion premium has almost disappeared.**
>
> **The question.** The board decides on Friday whether to keep weekend promotions. So it must see whether promotion weekends still out-earn ordinary weekends. It must also see whether that advantage is stable, growing, or fading.
>
> **Why this form.** This is a change-over-time comparison between two conditions, so it gets two lines. They show quarterly means of net takings (already net of markdowns) for promotion and ordinary weekends, with 95% confidence bands. Quarterly detail matches how the board reviews trading. It also smooths weekend-level noise without hiding the trend. In the recent quarters, the <!-- plain-language: everyday decision --> decision depends on the premium's size compared with the noise, so the bands stay on the slide.
>
> **Alternatives rejected.** The footfall-against-takings scatter (Chart C) shows promotion days sitting slightly higher, but it collapses time. So a fading premium and a stable one look identical, and the fade *is* the finding. A single pooled promo-vs-ordinary bar pair was rejected for the same reason. Two years of averaging would report a healthy premium that the latest quarters do not show.
>
> **AI draft vs my changes.** The [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate)'s Chart D had a boxed legend reading `promo=True`, default colours, a tinted panel, raw-unit axes, and a variable-name title. I applied the six declutter steps and named the series in words. I moved to an Okabe-Ito accent with grey context, annotated the early and late premium, and retitled it to the finding. (Session: [Prompt Diary](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-prompt-diary), 20 November.)

Compare your own rationale with the good and weak examples in the [design guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md), section "6. E: Explain".

## 🥇 Gold: The interactive alternative

This is one of many good answers. Part one is the same quarterly story, drawn as [hvPlot](https://hvplot.holoviz.org/) lines. Values appear on hover, and clicking the legend hides a line. The extension adds the daily weekend points below the quarterly means, and the reader can zoom in. Its one-sentence justification: *It lets a trading director answer "what about the Easter weekend?" with a mouse movement, in the session itself.* The cell below builds both.

In [ ]:
# 🥇 Gold model: An interactive version for the directors' working session
import hvplot.pandas  # noqa: F401

wk_plot = wk.assign(kind=np.where(wk["promo"], "Promotion weekend", "Ordinary weekend"))
quarterly_means = (wk_plot.groupby(["quarter", "kind"])["net_takings"]
                   .mean().round(1).reset_index())

quarterly = quarterly_means.hvplot.line(
    x="quarter", y="net_takings", by="kind",
    height=360, responsive=True, min_width=560, fontsize={"title": "10pt"},
    ylabel="Mean weekend net takings (£k)", xlabel="",
    title="Weekend takings, promotion vs ordinary: Hover for values, click legend to mute",
)

# Rows sorted by kind and kept in that order (sort_date=False), so each kind has the line chart's colour
points = wk_plot.sort_values("kind").hvplot.scatter(
    x="date", y="net_takings", by="kind", alpha=0.35, size=15, sort_date=False,
    height=360, responsive=True, min_width=560, fontsize={"title": "10pt"},
    ylabel="Daily net takings (£k)", xlabel="",
    title="Every weekend day, Jul 2024–Jun 2026: Zoom into any quarter",
)

(quarterly + points).cols(1)

**A model three-sentence argument:**

> In the working session, the interactive version answers the directors' unexpected questions ("which weekends?", "what about December?") by hover and zoom, on the spot. Interactivity suits exploration: The directors are investigating, so they should be able to hover and zoom themselves. In the board pack, the slide makes one fixed argument, so the static figure ships and the interactive version waits in the appendix for questions.

## 🚩 Red Flags: The post-mortem

The Associate's "declutter" deleted the **legend** without adding direct labels, deleted the **95% bands**, and then wrote a title the data does not support.

In business terms, the board can no longer tell which line is which, or how sure the numbers are.

**The naming.** With no legend and no labels, the board sees two anonymous lines. A board member cannot tell which is which without asking, so the slide does not answer the board's question. The deletion breaks step 4 of the declutter sequence ([design guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md), section "3. R: Reduce").

**The certainty.** The bands are the only part of the figure that shows how sure we are. Without them, two separated lines read as a steady premium the board can count on. With them, the last three quarters show the premium shrinking from about £6k to under £2k. In the last two quarters the bands overlap. By spring 2026 the numbers cannot tell a small premium from no premium at all. The title ("keep them") rests on evidence the slide does not show. If the board renews a two-year markdown budget because of that slide, the deletions could cost the company money. The [design guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md), section "5. C: Context", names deleted uncertainty the most serious mistake on its list.

**The catch:** Audit the information before and after any declutter. If a deletion changes what a careful reader concludes, it removed data.

**The fix**, made by directing your Associate: Restore the series naming and the 95% bands. Then retitle the slide to what the last four quarters support. The Bronze model above is the corrected figure, and the Silver model carries the corrected title. One acceptable prompt:

> *"Your declutter deleted the series naming and the 95% bands. Restore the bands and replace the legend with direct end-of-line labels. Then retitle: The premium has been shrinking for a year and is now within the margin of error. So the title cannot say 'keep them'."*

## 🤖 AI failure gallery

Associates often get *this week's* material wrong in the ways below.

### 1. The declutter that deletes data

**The plausible output.** Asked to "declutter this chart for a slide", the Associate returns minimal code. The diff, the line-by-line list of what changed, shows `ax.legend(...)`, `fill_between(...)`, and the error bars gone. Sometimes the Associate adds a comment: `# removed visual clutter`.

**Why it is wrong.** A legend deleted without replacement deletes the naming. Bands and error bars are findings about certainty. A slide stripped of uncertainty reads as more conclusive than the analysis behind it.

**The catch.** Audit every declutter as a diff. Name each thing removed and classify it as noise or data. If it answered a question (which line? how sure?), it was data.

### 2. The helpful truncation

**The plausible output.** "Can you make the difference between the branches clearer?" The Associate sets `ax.set_ylim(89, 95)` on a **bar** chart. The bars now look very different from each other. The code runs without errors. The Associate calls it "improved readability".

**Why it is wrong.** Bars show value as length. So a cut-off baseline turns a 4-point spread on a 100-point scale into a picture that looks like a 3-to-1 gap. A budget or bonus <!-- plain-language: everyday decision --> decision based on that picture rests on a distorted view.

**The catch.** A bar chart starts at zero, or at 1 for ratios on a log scale ([design guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md), section "5. C: Context"). If fine differences matter, switch to a dot plot with a stated non-zero scale.

### 3. "Colour-blind safe", says the machine

**The plausible output.** *"Is this red/green scheme okay for accessibility?"* *"Yes: Red and green differ in brightness here, so the chart remains readable for colour-blind users."* The Associate delivers this in a reassuring tone, sometimes with a made-up citation.

**Why it is wrong.** Red-green is the most common colour-vision deficiency ([design guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md), section "4. A: Accessible"). Someone with a vote in Marlow & Finch's boardroom probably has it. If colour is the only channel separating "promotion" from "ordinary", as in the quest's Chart C, part of the audience sees a one-series chart.

**The catch.** Test the palette yourself:

- Ask for a named safe palette: Okabe-Ito or ColorBrewer's colour-blind-safe sets <!-- docs-lint: allow: ColorBrewer is a proper noun -->
- Add a second channel: Direct labels, markers, or line style
- Apply the greyscale test: Imagine it printed in grey and check that the finding is still clear

### 4. Alt text that describes the furniture, and titles that outrun the data

**The plausible output.** Asked for alt text: *"A line chart with two coloured lines and shaded regions, with quarters on the x-axis and values on the y-axis."* Asked for a punchier title: *"Weekend promotions drive strong takings growth"*.

**Why it is wrong.** A screen-reader user learns only the figure's shape, so the figure gives them no evidence. The title turns a shrinking, possibly zero premium into "strong growth". The board minutes may then quote that overclaim.

**The catch.** For alt text, test against the formula *type + variables + takeaway*. If the sentence fits any chart of this data, it has no takeaway. For titles, ask whether a doubtful reader holding only this figure could defend the sentence. If not, rewrite the title yourself.

## Closing the week

Update your `QuestLog.md` if anything here changed your answers, and note what changed. Next week is **the [Gallery](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-gallery)**: Your group's figures go on the wall for a [Four-D crit](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-four-d-crit), with feedback for each [TRACE](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-trace) letter ([Week 9 Introduction](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/09/Introduction.ipynb)). Week 9's figures go in the group report.